In [17]:
# !git clone -b q0q https://github.com/IssacAnand/Plants-have-cancer.git
# %cd /content/Plants-have-cancer/backend
# !ls

In [18]:
# !git config pull.rebase false
# !git config --global user.email "wanqiinng@gmail.com"
# !git config --global user.name "wanq27"
# !git pull origin backend

## Load Dataset

### Imports

In [19]:
import pandas as pd
from sklearn.model_selection import train_test_split
import torch
from torch.utils.data import Dataset, DataLoader
import re
import json

In [20]:
import os
print("Current path:", os.getcwd())

dataset = "./Plants-have-cancer/backend/data/text/plantwild.json"
df=pd.read_json(dataset)
print(df.head(5))

Current path: /content
                                     apple black rot  \
0  Apple black rot typically appears as circular,...   
1  Apple black rot causes circular, sunken, black...   
2  Apple black rot appears as circular, sunken, a...   
3  Apple black rot appears as circular, sunken, b...   
4  Apple black rot appears as circular, sunken, b...   

                                          apple leaf  \
0  Apple leaf appearance is typically green, oval...   
1  Wilting, yellowing, and curling of apple leave...   
2  Healthy apple leaves are typically oval in sha...   
3  A healthy apple leaf appears green, shiny, and...   
4  The apple leaf may exhibit spots, discoloratio...   

                                  apple mosaic virus  \
0  Apple mosaic virus causes green and yellow mos...   
1  Apple mosaic virus causes mottling and distort...   
2  Apple mosaic virus causes light and dark green...   
3  Distinct mosaic-like patterns of yellow and gr...   
4  Apple mosaic virus c

## Preprocessing

In [21]:
def remove_disease_name(row):
    disease = str(row["disease"]).strip()
    description = str(row["description"]).strip()

    if not disease or not description:
        return description

    cleaned = re.sub(re.escape(disease), "", description, flags=re.IGNORECASE)
    cleaned = re.sub(r"\s+", " ", cleaned).strip(" ,.-:;")
    return cleaned


def preprocess_text_data(path, label_map):
    df = pd.read_json(path)

    # reshape wide -> long
    df_long = df.melt(var_name="disease", value_name="description")

    # clean
    df_long = df_long.dropna()
    df_long["description"] = df_long["description"].astype(str).str.strip()
    df_long = df_long[df_long["description"] != ""]

    # remove disease from descriptions
    df_long["description"] = df_long.apply(remove_disease_name, axis=1)

    # create numeric label from existing map label
    df_long["label"] = df_long["disease"].map(label_map)
    df_long = df_long.dropna(subset=["label"])
    df_long["label"] = df_long["label"].astype(int)

    return df_long

In [22]:
class PlantTextDataset(Dataset):
  def __init__(self, df, tokenizer, max_length=128):
    self.samples=df[["description", "label"]].values.tolist()
    self.tokenizer=tokenizer
    self.max_length=max_length

    print(df.head())

  def __len__(self):
    return len(self.samples)

  def __getitem__(self, index):
    text, label = self.samples[index]
    encoding = self.tokenizer(
        text,
        padding="max_length",
        truncation=True,
        max_length=self.max_length,
        return_tensors="pt"
    )

    return {
        "input_ids": encoding["input_ids"].squeeze(0),
        "attention_mask": encoding["attention_mask"].squeeze(0),
        "label": torch.tensor(label, dtype=torch.long)
    }

# Import Uncased BERT model from HuggingFace

In [23]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer

model_name = "recobo/agriculture-bert-uncased"

#initialize tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name)

#initialize model
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=89
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: recobo/agriculture-bert-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
bert.embeddings.position_ids               | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
bert.pooler.dense.bias                     | MISSING    | 
bert.pooler.dense.weight                   | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if 

## Prepare train and test datasets

In [25]:
#load label map
with open("./Plants-have-cancer/backend/data/label_map.json", "r") as f:
  label_map = json.load(f)

#initialize preprocessed dataset
preprocessed_df = preprocess_text_data(dataset, label_map)

train_df, test_df = train_test_split(
    preprocessed_df,
    test_size=0.2,
    stratify=preprocessed_df["label"],
    random_state=42
    )

train_dataset = PlantTextDataset(train_df, tokenizer, max_length=128)
test_dataset = PlantTextDataset(test_df, tokenizer, max_length=128)

print(train_df.head())
print(test_df.head())

sample = train_dataset[0]
print(sample["input_ids"].shape)
print(sample["attention_mask"].shape)
print(sample["label"])

                            disease  \
984    cabbage alternaria leaf spot   
948                   broccoli leaf   
1434                    cherry leaf   
4457  tomato yellow leaf curl virus   
582               bean mosaic virus   

                                            description  label  
984   presents as circular, dark brown lesions with ...     19  
948   Healthy broccoli leaves are dark green and hav...     18  
1434  shows characteristic yellow-brown spots or les...     27  
4457  causes yellowing and curling of tomato leaves,...     87  
582   causes mottling, yellowing, and curling of the...     11  
                        disease  \
1488           cherry leaf spot   
140          apple mosaic virus   
449                  basil leaf   
4444  tomato septoria leaf spot   
1865  corn northern leaf blight   

                                            description  label  
1488  presents as circular, purple lesions with yell...     28  
140   appears as mottling or mosai

## Define Training Arguments

In [26]:
print("Torch version: ", torch.__version__)
print("Cuda GPU is available: ", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device count:", torch.cuda.device_count())
    print("GPU name:", torch.cuda.get_device_name(0))

device = "cuda" if torch.cuda.is_available() else "cpu"
device

Torch version:  2.10.0+cu128
Cuda GPU is available:  True
Device count: 1
GPU name: Tesla T4


'cuda'

In [27]:
model_name = model_name.split("/")[-1]

args = TrainingArguments(
    f"./snapshots/{model_name}-finetuned",
    eval_strategy = "epoch",
    save_strategy = "epoch",
    save_total_limit = 10,
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=10,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model='f1',
    push_to_hub=False,
    fp16=True
)

## Define Evaluation Metrics

In [28]:
# !pip install evaluate
import evaluate
import numpy as np


metric_f1 = evaluate.load('f1')
metric_acc = evaluate.load('accuracy')

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=-1)
    f1 = metric_f1.compute(predictions=predictions, references=labels, average="macro")
    acc = metric_acc.compute(predictions=predictions, references=labels)
    return {**f1, **acc}

## Training

In [34]:
# from transformers import DataCollatorWithPadding
# data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

model.to(device)

trainer = Trainer(
    model,
    args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics
)

In [35]:
train_log = trainer.train()
metrics = trainer.evaluate()
print(metrics)

# save the best model
trainer.save_model("./best_model")
tokenizer.save_pretrained("./best_model")

Epoch,Training Loss,Validation Loss,F1,Accuracy
1,No log,1.280939,0.645478,0.647191
2,No log,1.177090,0.666856,0.673034
3,0.831445,1.114286,0.679655,0.680899
4,0.831445,1.100307,0.692313,0.695506
5,0.626203,1.096587,0.697784,0.702247
6,0.626203,1.121629,0.691565,0.692135
7,0.413514,1.125629,0.697954,0.698876
8,0.413514,1.143380,0.699825,0.700000
9,0.292065,1.152747,0.691219,0.692135
10,0.292065,1.146625,0.692651,0.693258


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La

{'eval_loss': 1.143377661705017, 'eval_f1': 0.699572162872477, 'eval_accuracy': 0.7, 'eval_runtime': 1.9303, 'eval_samples_per_second': 461.079, 'eval_steps_per_second': 29.012, 'epoch': 10.0}


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./best_model/tokenizer_config.json', './best_model/tokenizer.json')

## Evaluate

In [36]:
import numpy as np

pred_output = trainer.predict(test_dataset)
logits = pred_output.predictions
y_pred = np.argmax(logits, axis=-1)
y_true = pred_output.label_ids

id_to_label = {v: k for k, v in label_map.items()}

pred_names = [id_to_label[int(x)] for x in y_pred]
true_names = [id_to_label[int(x)] for x in y_true]

f1 = metric_f1.compute(predictions=y_pred, references=y_true, average="macro")
acc = metric_acc.compute(predictions=y_pred, references=y_true)

print("F1:", f1)
print("Accuracy:", acc)

F1: {'f1': 0.699572162872477}
Accuracy: {'accuracy': 0.7}


In [37]:
print(metrics)

{'eval_loss': 1.143377661705017, 'eval_f1': 0.699572162872477, 'eval_accuracy': 0.7, 'eval_runtime': 1.9303, 'eval_samples_per_second': 461.079, 'eval_steps_per_second': 29.012, 'epoch': 10.0}


In [38]:
results_df = pd.DataFrame({
    'True Label': true_names,
    'Predicted Label': pred_names,
    'Description': test_df['description'].reset_index(drop=True)
})
display(results_df.head(10))

,True Label,Predicted Label,Description
0,cherry leaf spot,cherry leaf spot,"presents as circular, purple lesions with yell..."
1,apple mosaic virus,apple mosaic virus,appears as mottling or mosaic-like patterns on...
2,basil leaf,basil leaf,"The appears green, smooth, and has a distincti..."
3,tomato septoria leaf spot,potato early blight,"presents as small, dark brown spots with a con..."
4,corn northern leaf blight,corn northern leaf blight,"appears as elongated, cigar-shaped lesions wit..."
5,potato late blight,potato late blight,"appears as dark, water-soaked lesions on leave..."
6,strawberry leaf scorch,garlic leaf blight,"causes brown necrotic areas on the leaves, fol..."
7,bean mosaic virus,bean mosaic virus,"causes mottled, yellowish-green patches on bea..."
8,corn northern leaf blight,corn gray leaf spot,Gray-green lesions with irregular borders and ...
9,citrus canker,citrus canker,"causes raised corky lesions on leaves, stems, ..."


## Top Misclassified Samples

In [39]:
# Build label_descriptions_df from plantwild.json
raw_df = pd.read_json(dataset)
label_descriptions_df = pd.DataFrame([
    {"Disease Name": disease, "Sample Description": raw_df[disease].dropna().iloc[0] if not raw_df[disease].dropna().empty else ""}
    for disease in raw_df.columns
])

# Misclassification analysis
misclassified_df = results_df[results_df['True Label'] != results_df['Predicted Label']]

# Merge to get the generic description for the True Label
misclassified_df = pd.merge(misclassified_df,
                            label_descriptions_df[['Disease Name', 'Sample Description']],
                            left_on='True Label', right_on='Disease Name',
                            suffixes=('', '_TrueLabel'))
misclassified_df = misclassified_df.drop(columns=['Disease Name'])
misclassified_df = misclassified_df.rename(columns={'Sample Description': 'True Label Description'})

# Merge to get the generic description for the Predicted Label
misclassified_df = pd.merge(misclassified_df,
                            label_descriptions_df[['Disease Name', 'Sample Description']],
                            left_on='Predicted Label', right_on='Disease Name',
                            suffixes=('', '_PredictedLabel'))
misclassified_df = misclassified_df.drop(columns=['Disease Name'])
misclassified_df = misclassified_df.rename(columns={'Sample Description': 'Predicted Label Description'})

misclassification_counts = misclassified_df.groupby(['True Label', 'Predicted Label']).size().reset_index(name='Count')
top_misclassifications_summary = misclassification_counts.sort_values(by='Count', ascending=False).head(10)
display(top_misclassifications_summary)

print("\nTop 10 individual misclassified samples with descriptions:")
pd.set_option('display.max_colwidth', None)
display(misclassified_df.head(10))
pd.reset_option('display.max_colwidth')

,True Label,Predicted Label,Count
49,cauliflower alternaria leaf spot,cabbage alternaria leaf spot,5
187,tomato mosaic virus,tobacco mosaic virus,5
84,cucumber powdery mildew,squash powdery mildew,4
94,ginger leaf spot,bell pepper leaf spot,4
79,cucumber leaf,bean leaf,4
151,rice sheath blight,ginger sheath blight,4
61,cherry leaf,bean leaf,3
18,bean leaf,potato leaf,3
121,lettuce mosaic virus,apple mosaic virus,3
175,tomato early blight,potato early blight,3



Top 10 individual misclassified samples with descriptions:


,True Label,Predicted Label,Description,True Label Description,Predicted Label Description
0,tomato septoria leaf spot,potato early blight,"presents as small, dark brown spots with a concentric ring pattern on the leaves","Tomato septoria leaf spot appears as numerous small, dark brown spots with yellow halos on leaves.",Potato early blight shows distinct concentric rings with dark brown lesions on leaves and stems.
1,strawberry leaf scorch,garlic leaf blight,"causes brown necrotic areas on the leaves, followed by browning and death of the entire leaf",Strawberry leaf scorch appears as reddish-brown necrotic lesions that often have a purple border.,"Garlic leaf blight appears as brown and yellow lesions on the leaves, eventually leading to their withering."
2,corn northern leaf blight,corn gray leaf spot,Gray-green lesions with irregular borders and dark green to brown spores on corn leaves indicate northern leaf blight,"Corn northern leaf blight appears as elongated, cigar-shaped lesions with tan centers and dark borders on corn leaves.",Corn gray leaf spot appears as small gray to tan lesions with dark borders on the leaves.
3,potato leaf,tomato leaf,"The appears wilted, yellowing, and may have brown lesions or spots","Yellowing and wilting of leaves with brown spots and necrotic lesions, often starting from the edges.","Tomato leaf may appear yellow or wilted, with brown or black spots or lesions, and curling or twisting."
4,cauliflower leaf,potato leaf,"can appear yellowing or brown with wilting and chlorotic spots, indicating a possible disease","Cauliflower leaves have a dense, curd-like appearance with pale green coloration and a smooth surface.","Yellowing and wilting of leaves with brown spots and necrotic lesions, often starting from the edges."
5,lettuce mosaic virus,tobacco mosaic virus,shows distinct mosaic patterns of light and dark green on the leaves,Lettuce mosaic virus causes mosaic patterns on the leaves with light and dark green patches.,Tobacco mosaic virus causes distinct light and dark green mottling or mosaics on the leaves.
6,peach leaf,strawberry leaf,"appears green with serrated edges and prominent veins, often featuring dark spots or discoloration","Peach leaf appears green and oval-shaped, with serrated edges and prominent veins running through it.",The strawberry leaf appears green with serrated edges and a sprawling shape.
7,broccoli leaf,celery leaf,The appears green and leafy with a large central stalk and dense clusters of rounded leaves,"Healthy broccoli leaf is green, lacy, and divided into smaller leaflets with a smooth and even texture.","The celery leaf may appear yellow or brown, with wilting or necrosis, indicating possible disease."
8,cucumber leaf,cauliflower leaf,"The appears pale green with yellow spots and a wilted, drooping appearance","The leaves of cucumber plants affected by disease may develop yellow, brown, or black spots and patches.","Cauliflower leaves have a dense, curd-like appearance with pale green coloration and a smooth surface."
9,strawberry leaf,potato leaf,"can appear with brown-black lesions, yellowing, and wilting, due to various diseases and pathogens",The strawberry leaf appears green with serrated edges and a sprawling shape.,"Yellowing and wilting of leaves with brown spots and necrotic lesions, often starting from the edges."


## Extract Feature Vector for MLP

In [40]:
print(f"Number of classification labels (output dimension): {model.config.num_labels}")
print(f"Hidden size of the base BERT model (feature vector dimension): {model.config.hidden_size}")


Number of classification labels (output dimension): 89
Hidden size of the base BERT model (feature vector dimension): 768


In [48]:
class FeatureExtractorModel(torch.nn.Module):
    def __init__(self, original_model):
        super().__init__()
        self.bert = original_model.bert

    def forward(self, input_ids, attention_mask):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        # Typically, the [CLS] token's embedding is used as the sentence representation
        # It's the first token (index 0) of the last hidden state
        cls_embedding = outputs.last_hidden_state[:, 0, :]
        return cls_embedding

def extract_features(dataset, model, device, batch_size=16):
    """Extract [CLS] token embeddings from BERT for all samples in a dataset."""
    feature_model = FeatureExtractorModel(model)
    feature_model.to(device)
    feature_model.eval()
    all_features = []
    all_labels = []
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=False)  # shuffle=False is critical

    with torch.no_grad():
        for batch in dataloader:
          input_ids = batch['input_ids'].to(device)
          attention_mask = batch['attention_mask'].to(device)
          labels = batch['label']
          features = feature_model(input_ids=input_ids, attention_mask=attention_mask)
          all_features.append(features.cpu())
          all_labels.append(labels)

    return torch.cat(all_features, dim=0), torch.cat(all_labels, dim=0)

model = AutoModelForSequenceClassification.from_pretrained("./best_model")
tokenizer = AutoTokenizer.from_pretrained("./best_model")
model.to(device)
model.eval()

train_text_features, train_labels = extract_features(train_dataset, model, device)
test_text_features, test_labels   = extract_features(test_dataset,  model, device)

print(f"Train text features: {train_text_features.shape}")  # [N_train, 768]
print(f"Test text features:  {test_text_features.shape}")   # [N_test, 768]

save_directory = "./Plants-have-cancer/backend/checkpoints/agriculture-bert-uncased/"

# create the directory if it doesn't exist
os.makedirs(save_directory, exist_ok=True)

# save for handoff
torch.save({
"train_features": train_text_features,
"train_labels":   train_labels,
"test_features":  test_text_features,
"test_labels":    test_labels,
}, os.path.join(save_directory, "bert_features.pt"))

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Train text features: torch.Size([3560, 768])
Test text features:  torch.Size([890, 768])


In [43]:
print(f"Shape of train_df: {train_df.shape}")
print(f"Shape of test_df: {test_df.shape}")

Shape of train_df: (3560, 3)
Shape of test_df: (890, 3)
